# 🧪 json2quantum
**Convierte un circuito cuántico definido en JSON a un circuito Qiskit.**

### Formato JSON esperado
```json
{
  "level": 6,
  "qubits": 4,
  "basis": ["CNOT", "Ry", "Rz", "X"],
  "operations": [
    {
      "gate": {
        "label": "elemental_rz",
        "name": "Rz",
        "target": 2,
        "parameter": 0.7854,
        "controls": []
      }
    }
  ]
}
```

## 📦 1. Instalación de dependencias

In [2]:
!pip install qiskit
!pip install pylatexenc

In [3]:
# Instalar Qiskit si no está disponible (necesario en Colab)
try:
    import qiskit
    print(f"✅ Qiskit {qiskit.__version__} ya instalado")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "qiskit", "qiskit-aer", "pylatexenc", "-q"])
    print("✅ Qiskit instalado correctamente")

✅ Qiskit 2.4.1 ya instalado


## 📚 2. Imports

In [7]:
import json
import math
from pathlib import Path

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import RZGate, RYGate, XGate, CXGate, HGate, ZGate, SGate, TGate
from qiskit.visualization import plot_histogram

print("✅ Imports OK")

✅ Imports OK


## 🔧 3. Parser JSON → Qiskit

Mapeamos cada `name` del JSON a la compuerta Qiskit correspondiente.
Las compuertas parametrizadas (Rz, Ry) toman el campo `parameter` en radianes.
Las compuertas controladas (CNOT) usan el campo `controls`.

In [ ]:
# ──────────────────────────────────────────────
# Mapa de nombres JSON → instrucción Qiskit
# ──────────────────────────────────────────────
GATE_MAP = {
    # Compuertas de 1 qubit — sin parámetro
    "X":    lambda p: XGate(),
    "H":    lambda p: HGate(),
    "Z":    lambda p: ZGate(),
    "S":    lambda p: SGate(),
    "T":    lambda p: TGate(),
    # Compuertas de 1 qubit — con parámetro (ángulo en radianes)
    "Rz":   lambda p: RZGate(p),
    "RZ":   lambda p: RZGate(p),
    "Ry":   lambda p: RYGate(p),
    "RY":   lambda p: RYGate(p),
    # Compuertas controladas
    "CNOT": lambda p: CXGate(),
    "CX":   lambda p: CXGate(),
}


def parse_json_to_circuit(json_data: dict) -> QuantumCircuit:
    """
    Convierte un diccionario con el formato json2quantum
    en un QuantumCircuit de Qiskit.

    Parámetros
    ----------
    json_data : dict
        Datos cargados desde el JSON.

    Retorna
    -------
    QuantumCircuit
        Circuito Qiskit listo para simular o visualizar.
    """
    n_qubits   = json_data["qubits"]
    operations = json_data["operations"]
    level      = json_data.get("level", "?")
    basis      = json_data.get("basis", [])

    print(f"📋 Nivel          : {level}")
    print(f"⚛️  Qubits         : {n_qubits}")
    print(f"🔑 Basis           : {basis}")
    print(f"🔢 Operaciones     : {len(operations)}")
    print("-" * 40)

    qr = QuantumRegister(n_qubits, name="q")
    cr = ClassicalRegister(n_qubits, name="c")
    qc = QuantumCircuit(qr, cr, name=f"nivel{level}")

    for i, op in enumerate(operations):
        gate_def = op["gate"]

        name      = gate_def["name"]
        target    = gate_def["target"]
        parameter = gate_def.get("parameter", None)   # None si no existe
        controls  = gate_def.get("controls", [])       # lista de {"qubit": k}

        # ── validar nombre ──────────────────────────
        if name not in GATE_MAP:
            raise ValueError(
                f"Operación {i}: compuerta '{name}' no reconocida. "
                f"Disponibles: {list(GATE_MAP.keys())}"
            )

        gate = GATE_MAP[name](parameter)
        gate.label = label   # etiqueta visual en el diagrama

        # ── construir lista de qubits [controles..., target] ──
        control_qubits = [qr[c["qubit"]] for c in controls]
        all_qubits     = control_qubits + [qr[target]]

        qc.append(gate, all_qubits)

        ctrl_str = [c["qubit"] for c in controls]
        param_str = f", θ={parameter:.4f}" if parameter is not None else ""
        print(f"  [{i:02d}] {name:<6} | target=q[{target}]{param_str} | controls={ctrl_str} | label={label}")

    # Medición al final
    qc.measure(qr, cr)

    print("-" * 40)
    print(f"✅ Circuito construido: {qc.num_qubits} qubits, {qc.depth()} capas de profundidad")
    return qc


print("✅ Parser definido")

## 📂 4. Cargar el JSON y construir el circuito

In [12]:
# ── Cambia esta ruta al JSON que quieras parsear ──
JSON_FILE = "nivel6.json"

with open(JSON_FILE, "r") as f:
    data = json.load(f)

print("📄 JSON cargado:")
print(json.dumps(data, indent=2))

📄 JSON cargado:
{
  "level": 6,
  "qubits": 4,
  "basis": [
    "CNOT",
    "Ry",
    "Rz",
    "X"
  ],
  "operations": [
    {
      "gate": {
        "name": "Rz",
        "target": 2,
        "parameter": 0.7854,
        "controls": []
      }
    },
    {
      "gate": {
        "name": "CNOT",
        "target": 2,
        "controls": [
          {
            "qubit": 0
          }
        ]
      }
    }
  ]
}


In [ ]:
qc = parse_json_to_circuit(data)

## 🖼️ 5. Visualizar el circuito

In [ ]:
print("\n🔭 Diagrama del circuito:")
qc_draw = qc.copy()
qc_draw.remove_final_measurements()  # dibujar sin mediciones para mayor claridad

display(qc_draw.draw(output="mpl", style="iqp", fold=-1))

## 🚀 6. Simular con Aer

In [ ]:
SHOTS = 4096

simulator = AerSimulator()
job       = simulator.run(qc, shots=SHOTS)
result    = job.result()
counts    = result.get_counts()

print(f"🎲 Resultados ({SHOTS} shots):")
for state, count in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "█" * int(count / SHOTS * 40)
    print(f"  |{state}⟩ : {count:5d}  {bar}")

In [ ]:
fig = plot_histogram(
    counts,
    title=f"Distribución de mediciones — nivel {data.get('level', '?')}",
    figsize=(10, 4),
    color="#6929C4",
)
plt.tight_layout()
plt.show()

## 📊 7. Información del circuito

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator

backend = AerSimulator()
pm      = generate_preset_pass_manager(optimization_level=1, backend=backend)
qc_t    = pm.run(qc)

print("📊 Estadísticas del circuito")
print(f"  Profundidad (original)   : {qc.depth()}")
print(f"  Profundidad (transpilado): {qc_t.depth()}")
print(f"  Número de qubits         : {qc.num_qubits}")
print(f"  Operaciones              : {qc.count_ops()}")
print(f"  Operaciones transpiladas : {qc_t.count_ops()}")

---
## 🧩 8. Ejemplo: cargar desde string en lugar de archivo

Útil para pruebas rápidas sin necesitar un archivo en disco.

In [ ]:
inline_json = """
{
    "level": 99,
    "qubits": 2,
    "basis": ["H", "CNOT"],
    "operations": [
        {
            "gate": {
                "label": "hadamard_q0",
                "name": "H",
                "target": 0,
                "controls": []
            }
        },
        {
            "gate": {
                "label": "entangle",
                "name": "CNOT",
                "target": 1,
                "controls": [{"qubit": 0}]
            }
        }
    ]
}
"""

data_bell = json.loads(inline_json)
qc_bell   = parse_json_to_circuit(data_bell)

display(qc_bell.draw(output="mpl", style="iqp"))

# Simular
result_bell = AerSimulator().run(qc_bell, shots=2048).result()
plot_histogram(result_bell.get_counts(), title="Bell State |Φ+⟩", color="#00B28A")